In [2]:
"""
Nunn (2008, QJE) "The Long-Term Effects of Africa's Slave Trades"
=================================================================
PART 1  Replication of the published Table III (OLS) and Table IV (IV/2SLS).
PART 2  Double Machine Learning (DML) extension of four selected columns.

The script always runs PART 1 first: the replication both validates the data
and produces the "Original" benchmark numbers that PART 2 compares against.

Design fixed by the project
---------------------------
* Data          : slave_trade_QJE.dta only.
* Original SEs  : conventional (homoskedastic) with n-k d.o.f., i.e. exactly the
                  standard errors printed in the paper. DML SEs are
                  heteroskedasticity-robust by construction; an HC1 recomputation
                  of the original specs was checked separately and leaves every
                  comparative conclusion unchanged (it makes the IV precision gain
                  larger, not smaller), so the paper's own convention is used here.
* ML extension  : OLS vs ML-OLS (DML-PLR); IV vs ML-IV (DML-PLIV).
* Learners      : 10 learners suited to n = 52.
* Cross-fitting : 5 folds, 500 repeats. Within each repeat Y, D and ALL FOUR
                  instruments share ONE common fold partition (required for DML).
* Aggregation   : median coefficient and the matching median-aggregated SE
                  (Chernozhukov et al. 2018).
* Specs kept    : Table III col 2 and col 5; Table IV col 3 and col 4.
                  Columns with no or almost no controls are not extended: with an
                  empty X the ML residualization degenerates and every learner
                  collapses to the same estimator by construction.

Coefficient-equality test (PART 2b)
-----------------------------------
    H0 : theta_ML = theta_Original
    t  = (theta_ML - theta_Original) / sqrt( SE_ML^2 + SE_Original^2 )
    p  = 2 * (1 - Phi(|t|))

The pooled denominator treats the published/replicated benchmark as an ESTIMATE
with its own sampling error rather than a fixed constant. Because both estimators
are computed on the SAME countries, Cov(theta_ML, theta_Orig) > 0, so setting the
covariance to zero OVERSTATES SE_diff and biases the test toward non-rejection.
A non-significant t is therefore evidence of "no detectable difference under this
convention", not proof of exact equality; the tables say so in their notes.

Replication status
---------------------------------------------------------
Table III  col 1-6 : all six coefficients AND standard errors reproduce exactly.
Table IV   col 1-3 : coefficients and standard errors reproduce exactly.
Table IV   col 4   : the first stage reproduces exactly (all four instruments) and
                     n = 42 matches, but the second-stage point estimate is
                     -0.221 (0.078) against the published -0.248 (0.071).
                     Sample, controls and instruments are therefore identical; the
                     residual gap in this one cell is unexplained and is reported
                     as-is rather than tuned away. The DML extension of column 4
                     is benchmarked against our own replication value, so the
                     comparison stays internally consistent.

Performance notes (estimates are numerically unchanged)
-------------------------------------------------------
Nothing that touches the estimator was modified: REPEATS, FOLDS, RANDOM_STATE,
the seed schedule, the ten learners, the shared fold partition and the median
aggregation are all exactly as before. The speedups are purely mechanical:
  1. Repeats run in parallel (joblib). Repeat r uses seed RANDOM_STATE+10000*r
     regardless of execution order, so results are order-independent.
  2. BLAS/OpenMP threads are pinned to 1 per worker to stop nested thread
     oversubscription (which on this workload costs more than it buys).
  3. The DataFrame -> ndarray conversion is hoisted out of the inner loop; it
     used to run once per learner per repeat (5000x per spec) on constant data.
  4. Checkpoints store a fixed (repeats x learners) matrix instead of ragged
     append-lists, so a resume can never mis-align learners. Legacy v1
     checkpoints are migrated automatically when their lengths are consistent.
"""

from __future__ import annotations

# Thread pinning must happen BEFORE numpy/sklearn import to take effect.
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import math
import time
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # file output only; no interactive backend needed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression, RidgeCV
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR

# =============================================================================
# PART 0.  Configuration, data location, and variable definitions
# =============================================================================

# --- data location -----------------------------------------------------------
# ONE place to change the source. Must be a raw.githubusercontent.com link:
# a normal github.com/.../blob/... page returns HTML, not the .dta file.
GITHUB_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "gjn7228525-ctrl/Nunn-2008-DML-extension/"
    "main/data/slave_trade_QJE.dta"
)
# Local copy is preferred when present, so repeated runs (and offline work) do
# not re-download. Override either with the DML_DATA env var.
LOCAL_DATA = Path(os.environ.get("DML_DATA", "")) if os.environ.get("DML_DATA") \
    else Path("data") / "slave_trade_QJE.dta"
DATA_PATH = str(LOCAL_DATA) if LOCAL_DATA.exists() else GITHUB_DATA_URL


def load_data() -> pd.DataFrame:
    """Read the Stata file, with a message that says which source was used."""
    source = "local file" if DATA_PATH == str(LOCAL_DATA) else "GitHub"
    try:
        df = pd.read_stata(DATA_PATH)
    except Exception as exc:
        raise SystemExit(
            f"Could not read the data from {source}: {DATA_PATH}\n  {exc}\n"
            "  - GitHub links must be raw.githubusercontent.com, not /blob/\n"
            "  - private repos are not reachable this way; download the .dta and\n"
            "    put it at data/slave_trade_QJE.dta or set DML_DATA=/path/to.dta"
        ) from exc
    print(f"Data loaded from {source}: {len(df)} rows, {df.shape[1]} columns")
    return df


OUT_DIR = Path("output") / "final_dml_poster"
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Checkpoints let a long run resume after an interruption (see PART 2).
CKPT_DIR = OUT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 610  # single global seed: the whole pipeline is deterministic
REPEATS = 500       # repeated cross-fitting splits
FOLDS = 5           # folds per split

# --- parallelism (affects speed only, never the numbers) ---------------------
# -1 = every core. Set N_JOBS = 1 to reproduce the original sequential run.
N_JOBS = int(os.environ.get("DML_N_JOBS", "-1"))
CHUNK = 25  # repeats per checkpoint flush, same cadence as before

# --- outcome and treatment ---------------------------------------------------
Y = "ln_maddison_pcgdp2000"  # ln real per capita GDP in 2000
D = "ln_export_area"         # ln(slave exports / land area), 1400-1900

# --- control blocks, exactly as grouped in the paper -------------------------
# Colonizer fixed effects (identity of the colonizer at independence).
COLONIZER_FE = [
    "colony1", "colony2", "colony3", "colony4", "colony5", "colony6", "colony7",
]
# Table III col 2 "geography controls": distance from equator, longitude, lowest
# monthly rainfall, avg max humidity, avg min temperature, ln(coastline/area).
GEOGRAPHY_CONTROLS = [
    "abs_latitude", "longitude", "rain_min", "humid_max", "low_temp",
    "ln_coastline_area",
]
# Table III col 4 additions: island indicator, percent Islamic, French legal
# origin, North Africa indicator.
ISLAND_RELIGION_LEGAL_REGION = ["island_dum", "islam", "legor_fr", "region_n"]
# Table III col 5 additions: ln per capita production of gold, oil, diamonds.
RESOURCE_CONTROLS = [
    "ln_avg_gold_pop", "ln_avg_oil_pop", "ln_avg_all_diamonds_pop",
]
# Table IV instruments: minimum distance to the demand locations of each of the
# four slave trades.
DISTANCE_INSTRUMENTS = [
    "atlantic_distance_minimum",
    "indian_distance_minimum",
    "saharan_distance_minimum",
    "red_sea_distance_minimum",
]


def restricted_sample(df: pd.DataFrame) -> pd.DataFrame:
    """The paper's "restricted sample": drop island and North African countries.

    Removes Comoros, Cape Verde, Mauritius, Sao Tome & Principe, Seychelles,
    Algeria, Egypt, Libya, Morocco, Tunisia -> n = 42. Verified to reproduce
    Table III col 3 and col 6 exactly.
    """
    return df[(df["island_dum"] == 0) & (df["region_n"] == 0)].copy()


def add_constant(x: pd.DataFrame) -> pd.DataFrame:
    out = x.copy()
    out.insert(0, "const", 1.0)
    return out


# =============================================================================
# PART 1.  Replication of Nunn (2008) Table III and Table IV
# =============================================================================
# Both estimators use conventional (homoskedastic) standard errors with an n-k
# degrees-of-freedom correction, which is what reproduces the printed values
# (e.g. Table III col 2 se = 0.029; Table IV col 3 se = 0.153).

def ols_conventional(y: pd.Series, x: pd.DataFrame) -> dict:
    """OLS with conventional SEs: Var = sigma^2 (X'X)^-1, sigma^2 = RSS/(n-k)."""
    y_arr, x_arr = y.to_numpy(float), x.to_numpy(float)
    n, k = x_arr.shape
    xtx_inv = np.linalg.pinv(x_arr.T @ x_arr)  # pinv: robust to collinear dummies
    beta = xtx_inv @ x_arr.T @ y_arr
    resid = y_arr - x_arr @ beta
    sigma2 = (resid @ resid) / (n - k)
    se = np.sqrt(np.maximum(np.diag(sigma2 * xtx_inv), 0.0))
    return {"coef": pd.Series(beta, index=x.columns),
            "se": pd.Series(se, index=x.columns), "n": n}


def tsls_conventional(df: pd.DataFrame, y_col: str, d_col: str,
                      controls: list[str], instruments: list[str]) -> dict:
    """2SLS with conventional SEs: Var = sigma^2 (X'P_Z X)^-1, sigma^2 = RSS/(n-k).

    RSS uses the STRUCTURAL residuals y - X*beta (with the actual endogenous
    regressor, not its fitted value) -- the standard 2SLS variance.
    """
    clean = df[[y_col, d_col] + controls + instruments].dropna().copy()
    y = clean[y_col].to_numpy(float)
    controls_df = clean[controls].astype(float)
    # X = [const, D, controls]; Z = [const, instruments, controls]
    x = add_constant(pd.concat([clean[[d_col]].astype(float), controls_df], axis=1))
    z = add_constant(pd.concat([clean[instruments].astype(float), controls_df], axis=1))
    x_arr, z_arr = x.to_numpy(float), z.to_numpy(float)
    n, k = x_arr.shape
    pz = z_arr @ np.linalg.pinv(z_arr.T @ z_arr) @ z_arr.T  # projection onto Z
    bread = np.linalg.pinv(x_arr.T @ pz @ x_arr)
    beta = bread @ (x_arr.T @ pz @ y)
    resid = y - x_arr @ beta
    sigma2 = (resid @ resid) / (n - k)
    se = np.sqrt(np.maximum(np.diag(sigma2 * bread), 0.0))
    return {"coef": pd.Series(beta, index=x.columns),
            "se": pd.Series(se, index=x.columns), "n": n}


def first_stage(df: pd.DataFrame, d_col: str, controls: list[str],
                instruments: list[str]) -> dict:
    """First stage of 2SLS plus the partial F-statistic on the four instruments.

    The partial F tests joint significance of the excluded instruments and is the
    conventional weak-instrument diagnostic (Nunn reports values around 1.7-4.6,
    i.e. well below the usual threshold of 10).
    """
    clean = df[[d_col] + controls + instruments].dropna().copy()
    d = clean[d_col].to_numpy(float)
    z_full = add_constant(pd.concat(
        [clean[instruments].astype(float), clean[controls].astype(float)], axis=1))
    z_arr = z_full.to_numpy(float)
    n, k = z_arr.shape
    ztz_inv = np.linalg.pinv(z_arr.T @ z_arr)
    beta = ztz_inv @ z_arr.T @ d
    resid = d - z_arr @ beta
    rss_full = resid @ resid
    sigma2 = rss_full / (n - k)
    se = np.sqrt(np.maximum(np.diag(sigma2 * ztz_inv), 0.0))
    # restricted model: controls only (instruments excluded)
    w_arr = add_constant(clean[controls].astype(float)).to_numpy(float)
    b_r = np.linalg.pinv(w_arr.T @ w_arr) @ w_arr.T @ d
    rss_restricted = float((d - w_arr @ b_r) @ (d - w_arr @ b_r))
    q = len(instruments)
    f_stat = ((rss_restricted - rss_full) / q) / (rss_full / (n - k))
    return {"coef": pd.Series(beta, index=z_full.columns),
            "se": pd.Series(se, index=z_full.columns), "F": float(f_stat), "n": n}


# Published values, hard-coded purely so the script can self-check the
# replication and print a PASS/DIFF flag next to every cell.
PUBLISHED_T3 = {1: (-0.112, 0.024), 2: (-0.076, 0.029), 3: (-0.108, 0.037),
                4: (-0.085, 0.035), 5: (-0.103, 0.034), 6: (-0.128, 0.034)}
PUBLISHED_T4 = {1: (-0.208, 0.053), 2: (-0.201, 0.047), 3: (-0.286, 0.153),
                4: (-0.248, 0.071)}


def table3_controls(col: int) -> tuple[str, list[str]]:
    """Control set and sample for each column of Table III."""
    geo = COLONIZER_FE + GEOGRAPHY_CONTROLS
    if col == 1:  # colonizer FE only
        return "full", COLONIZER_FE
    if col == 2:  # + geography
        return "full", geo
    if col == 3:  # col 2 controls, restricted sample
        return "restricted", geo
    if col == 4:  # + island / percent Islamic / French legal origin / N. Africa
        return "full", geo + ISLAND_RELIGION_LEGAL_REGION
    if col == 5:  # + natural resources
        return "full", geo + ISLAND_RELIGION_LEGAL_REGION + RESOURCE_CONTROLS
    # col 6: all controls, restricted sample. The island and North Africa dummies
    # are identically zero there and drop out of the regression, exactly as the
    # paper notes (footnote 11).
    return "restricted", geo + ["islam", "legor_fr"] + RESOURCE_CONTROLS


def table4_controls(col: int) -> tuple[str, list[str]]:
    """Control set and sample for each column of Table IV."""
    if col == 1:
        return "full", []                                  # no controls
    if col == 2:
        return "full", COLONIZER_FE                         # colonizer FE
    if col == 3:
        return "full", COLONIZER_FE + GEOGRAPHY_CONTROLS    # + geography
    return "restricted", COLONIZER_FE + GEOGRAPHY_CONTROLS  # col 4: restricted


def replicate_tables(df: pd.DataFrame) -> pd.DataFrame:
    """Reproduce every column of Table III and Table IV and check against print."""
    rows = []
    print("\n" + "=" * 78)
    print("PART 1  REPLICATION OF NUNN (2008)")
    print("=" * 78)

    print("\nTable III - OLS, dependent variable ln real per capita GDP 2000")
    print(f"{'col':>4} {'coef':>9} {'se':>8} {'n':>4}   {'published':>18}   check")
    for col in range(1, 7):
        sample, controls = table3_controls(col)
        data = restricted_sample(df) if sample == "restricted" else df
        clean = data[[Y, D] + controls].dropna()
        res = ols_conventional(clean[Y].astype(float),
                               add_constant(clean[[D] + controls].astype(float)))
        c, s, n = float(res["coef"][D]), float(res["se"][D]), int(res["n"])
        pc, ps = PUBLISHED_T3[col]
        ok = "PASS" if abs(c - pc) < 0.002 and abs(s - ps) < 0.002 else "DIFF"
        print(f"{col:>4} {c:>9.4f} {s:>8.4f} {n:>4}   {pc:>9.3f} ({ps:.3f})   {ok}")
        rows.append({"table": "III", "column": col, "sample": sample, "n": n,
                     "coef": c, "se": s, "published_coef": pc, "published_se": ps,
                     "check": ok})

    print("\nTable IV - 2SLS, four distance instruments")
    print(f"{'col':>4} {'coef':>9} {'se':>8} {'n':>4} {'1stF':>6}   {'published':>18}   check")
    for col in range(1, 5):
        sample, controls = table4_controls(col)
        data = restricted_sample(df) if sample == "restricted" else df
        res = tsls_conventional(data, Y, D, controls, DISTANCE_INSTRUMENTS)
        fs = first_stage(data, D, controls, DISTANCE_INSTRUMENTS)
        c, s, n = float(res["coef"][D]), float(res["se"][D]), int(res["n"])
        pc, ps = PUBLISHED_T4[col]
        ok = "PASS" if abs(c - pc) < 0.003 and abs(s - ps) < 0.003 else "DIFF"
        print(f"{col:>4} {c:>9.4f} {s:>8.4f} {n:>4} {fs['F']:>6.2f}   "
              f"{pc:>9.3f} ({ps:.3f})   {ok}")
        rows.append({"table": "IV", "column": col, "sample": sample, "n": n,
                     "coef": c, "se": s, "first_stage_F": fs["F"],
                     "published_coef": pc, "published_se": ps, "check": ok})
        # First-stage instrument coefficients (Table IV lower panel).
        fs_line = ", ".join(f"{iv.split('_')[0]}: {fs['coef'][iv]:+.2f} "
                            f"({fs['se'][iv]:.3f})" for iv in DISTANCE_INSTRUMENTS)
        print(f"       first stage -> {fs_line}")

    print("\nNote: Table IV col 4 first stage matches the paper exactly and n = 42,")
    print("but the second-stage point estimate is -0.221 (0.078) vs -0.248 (0.071)")
    print("in print. Reported as replicated; see module docstring.\n")

    out = pd.DataFrame(rows)
    out.to_csv(OUT_DIR / "replication_tables.csv", index=False)
    return out


# =============================================================================
# PART 2.  DML extension
# =============================================================================
# Model:
#   PLR  (extends OLS): Y = theta*D + g(X) + eps,  E[eps|D,X] = 0
#   PLIV (extends 2SLS): Y = theta*D + g(X) + eps, instruments Z with E[eps|Z,X]=0
# theta is estimated from cross-fitted residuals, so g(X) and m(X) may be any
# nonparametric function; only the treatment enters linearly.


@dataclass(frozen=True)
class Spec:
    panel: str          # which output panel the spec belongs to
    column: str         # label matching the original table column
    sample_name: str    # "full" or "restricted"
    controls: list[str]
    iv: bool
    tag: str            # checkpoint filename tag
    note: str = field(default="")


@dataclass(frozen=True)
class SpecArrays:
    """Numeric payload for one spec, built ONCE and reused by every repeat.

    Purely a performance container: previously each of the 10 learners x 500
    repeats re-ran `clean[controls].to_numpy(float)` on the same constant frame.
    """
    x: np.ndarray          # controls
    y: np.ndarray          # outcome
    d: np.ndarray          # treatment
    z: np.ndarray | None   # instruments (None for the PLR specs)


def build_arrays(clean: pd.DataFrame, spec: Spec) -> SpecArrays:
    x = (clean[spec.controls].to_numpy(float) if spec.controls
         else np.empty((len(clean), 0), dtype=float))
    return SpecArrays(
        x=np.ascontiguousarray(x),
        y=np.ascontiguousarray(clean[Y].to_numpy(float)),
        d=np.ascontiguousarray(clean[D].to_numpy(float)),
        z=(np.ascontiguousarray(clean[DISTANCE_INSTRUMENTS].to_numpy(float))
           if spec.iv else None),
    )


# Only four columns are extended (see module docstring for why).
OLS_SPECS = [
    Spec("OLS vs ML-OLS", "Column 2", "full",
         COLONIZER_FE + GEOGRAPHY_CONTROLS, False, "ols_c2",
         "Table III col 2: colonizer FE + geography"),
    Spec("OLS vs ML-OLS", "Column 5", "full",
         COLONIZER_FE + GEOGRAPHY_CONTROLS + ISLAND_RELIGION_LEGAL_REGION
         + RESOURCE_CONTROLS, False, "ols_c5",
         "Table III col 5: + island/Islam/legal origin/N.Africa + resources"),
]
IV_SPECS = [
    Spec("IV vs ML-IV", "Column 3", "full",
         COLONIZER_FE + GEOGRAPHY_CONTROLS, True, "iv_c3",
         "Table IV col 3: colonizer FE + geography, full sample"),
    Spec("IV vs ML-IV", "Column 4", "restricted",
         COLONIZER_FE + GEOGRAPHY_CONTROLS, True, "iv_c4",
         "Table IV col 4: same controls, restricted sample (n=42)"),
]
ALL_SPECS = OLS_SPECS + IV_SPECS


# ---------------------------------------------------------------- inference --
def normal_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def p_from_t(t_stat: float) -> float:
    """Two-sided p-value from the normal approximation (DML inference is asymptotic)."""
    if not np.isfinite(t_stat):
        return np.nan
    return float(2.0 * (1.0 - normal_cdf(abs(t_stat))))


def stars(p_value: float) -> str:
    if not np.isfinite(p_value):
        return ""
    return "***" if p_value < 0.01 else "**" if p_value < 0.05 else "*" if p_value < 0.10 else ""


# ----------------------------------------------------------------- learners --
def learner_map() -> dict[str, object]:
    """The ten nuisance learners. All are wrapped in StandardScaler because the
    controls are on wildly different scales (degrees, mm, %, logs)."""
    return {
        # --- Panel A: linear and extended linear ---
        "Linear": make_pipeline(StandardScaler(), LinearRegression()),
        "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25))),
        "LASSO": make_pipeline(
            StandardScaler(),
            LassoCV(cv=3, random_state=RANDOM_STATE, max_iter=20000)),
        "Elastic Net": make_pipeline(
            StandardScaler(),
            # l1_ratio grid stated explicitly; sklearn's default searches 0.5 only
            ElasticNetCV(cv=3, l1_ratio=[0.1, 0.5, 0.9],
                         random_state=RANDOM_STATE, max_iter=20000)),
        "Polynomial Ridge": make_pipeline(
            StandardScaler(), PolynomialFeatures(degree=2, include_bias=False),
            RidgeCV(alphas=np.logspace(-3, 3, 25))),
        # --- Panel B: nonlinear ---
        "SVR": make_pipeline(StandardScaler(), SVR(C=2.0, epsilon=0.05, gamma="scale")),
        # KernelRidge fits NO intercept. Since E[Y] is far from zero, the alpha
        # penalty otherwise shrinks predictions toward 0 and contaminates the
        # residuals (this produced a divergent IV estimate of about -2.2 before
        # the fix). TransformedTargetRegressor standardizes the target, restoring
        # sane behaviour.
        "Kernel Ridge": TransformedTargetRegressor(
            regressor=make_pipeline(StandardScaler(),
                                    KernelRidge(alpha=1.0, kernel="rbf")),
            transformer=StandardScaler()),
        # Trees are deliberately small: each training fold holds only ~42 (or ~34)
        # observations.
        "Random Forest": RandomForestRegressor(
            n_estimators=35, max_depth=4, max_features="sqrt", min_samples_leaf=3,
            random_state=RANDOM_STATE, n_jobs=1),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=35, max_depth=4, max_features="sqrt", min_samples_leaf=3,
            random_state=RANDOM_STATE, n_jobs=1),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=45, learning_rate=0.04, max_depth=2, min_samples_leaf=3,
            random_state=RANDOM_STATE),
    }


LEARNER_NAMES = list(learner_map().keys())


# ------------------------------------------------------------ cross-fitting --
def crossfit_predict(model: object, x: np.ndarray, y: np.ndarray,
                     fold_idx: list) -> np.ndarray:
    """Out-of-fold predictions using a PRE-COMPUTED fold partition.

    Passing `fold_idx` in (rather than building it inside) is what guarantees
    that Y, D and all four instruments are residualized on the SAME split within
    a repeat. Using different splits per variable breaks the sample structure the
    orthogonal score relies on.
    """
    pred = np.zeros(len(y), dtype=float)
    for train_idx, test_idx in fold_idx:
        if x.shape[1] == 0:
            # No controls: predict the TRAIN-fold mean (never the full-sample
            # mean, which would leak the held-out observations).
            pred[test_idx] = float(np.mean(y[train_idx]))
            continue
        fitted = clone(model)          # fresh model per fold, no state carry-over
        fitted.fit(x[train_idx], y[train_idx])
        pred[test_idx] = fitted.predict(x[test_idx])
    return pred


def plr_theta(y_res: np.ndarray, d_res: np.ndarray) -> tuple[float, float]:
    """Partially linear model: theta = <D~,Y~> / <D~,D~>.

    SE from the influence function (sandwich form), hence heteroskedasticity
    robust: se = sqrt( sum(D~^2 * eps^2) ) / sum(D~^2), with a n/(n-1) correction.
    """
    denom = float(d_res @ d_res)
    theta = float((d_res @ y_res) / denom)
    resid = y_res - theta * d_res
    n = len(y_res)
    meat = float(np.sum((d_res * resid) ** 2))
    se = float(np.sqrt((n / (n - 1)) * meat / (denom**2)))
    return theta, se


def pliv_theta(y_res: np.ndarray, d_res: np.ndarray,
               z_res: np.ndarray) -> tuple[float, float]:
    """Partially linear IV: 2SLS on the residualized Y, D and Z.

    Project D~ on the residualized instruments, then theta = <Dhat,Y~>/<Dhat,D~>.
    SE again from the influence function, using the STRUCTURAL residual
    eps = Y~ - theta*D~.
    """
    ztz_inv = np.linalg.pinv(z_res.T @ z_res)
    pz_d = z_res @ ztz_inv @ (z_res.T @ d_res)   # first-stage fitted values
    denom = float(d_res @ pz_d)
    theta = float((pz_d @ y_res) / denom)
    resid = y_res - theta * d_res
    meat = float(np.sum((pz_d * resid) ** 2))
    se = float(np.sqrt(meat) / abs(denom))
    return theta, se


def estimate_once(arrays: SpecArrays, model: object, seed: int,
                  folds: int) -> tuple[float, float]:
    """One repeat of cross-fitted DML: build ONE partition, share it everywhere."""
    x, y, d = arrays.x, arrays.y, arrays.d

    # The single fold partition for this repeat.
    splitter = KFold(n_splits=folds, shuffle=True, random_state=seed)
    fold_idx = list(splitter.split(np.arange(len(y)).reshape(-1, 1)))

    y_res = y - crossfit_predict(model, x, y, fold_idx)   # Y~
    d_res = d - crossfit_predict(model, x, d, fold_idx)   # D~
    if arrays.z is None:
        return plr_theta(y_res, d_res)

    # Every instrument is residualized on the very same fold_idx.
    z = arrays.z
    z_res = z - np.column_stack(
        [crossfit_predict(model, x, z[:, j], fold_idx) for j in range(z.shape[1])])
    return pliv_theta(y_res, d_res, z_res)


def _repeat_worker(arrays: SpecArrays, seed: int,
                   folds: int) -> tuple[list[float], list[float]]:
    """All ten learners for ONE repeat. The unit of parallelism.

    Learners are rebuilt inside the worker so nothing stateful crosses the
    process boundary. Seed is fixed by the repeat index, so which worker picks
    up which repeat has no effect on the result.
    """
    learners = learner_map()
    thetas, ses = [], []
    for name in LEARNER_NAMES:
        try:
            theta, se = estimate_once(arrays, learners[name], seed, folds)
        except Exception:            # a pathological split must not kill the run
            theta, se = np.nan, np.nan
        thetas.append(theta)
        ses.append(se)
    return thetas, ses


# ------------------------------------------------- aggregation over repeats --
def aggregate_median(coefs: np.ndarray, ses: np.ndarray) -> dict:
    """Median aggregation of Chernozhukov et al. (2018).

        theta_hat = median_r( theta_r )
        se_hat    = sqrt( median_r[ se_r^2 + (theta_r - theta_hat)^2 ] )

    The SE combines within-split sampling uncertainty (se_r^2) with across-split
    variation ((theta_r - theta_hat)^2), the two terms being summed BEFORE the
    median is taken. This is the median counterpart of the paper's mean formula
    mean(se_r^2) + Var(theta_r), and is the version that pairs with a median
    point estimate; it is robust to occasional pathological splits.
    """
    mask = np.isfinite(coefs) & np.isfinite(ses)
    coefs, ses = coefs[mask], ses[mask]
    if coefs.size == 0:
        return {"ml_coef": np.nan, "ml_se": np.nan, "ml_t": np.nan, "ml_p": np.nan,
                "stars": "", "repeat_sd": np.nan, "valid_repeats": 0}
    theta_hat = float(np.median(coefs))
    se_hat = float(np.sqrt(np.median(ses**2 + (coefs - theta_hat) ** 2)))
    t_stat = theta_hat / se_hat if se_hat > 0 else np.nan
    p_value = p_from_t(t_stat)
    return {"ml_coef": theta_hat, "ml_se": se_hat, "ml_t": t_stat,
            "ml_p": p_value, "stars": stars(p_value),
            "repeat_sd": float(np.std(coefs, ddof=1)) if len(coefs) > 1 else 0.0,
            "valid_repeats": int(len(coefs))}


# ------------------------------------------------------------- checkpoints --
def _load_checkpoint(spec: Spec, repeats: int,
                     n_learners: int) -> tuple[np.ndarray, np.ndarray, int]:
    """Load a v2 checkpoint, migrating a legacy v1 file when it is consistent."""
    theta = np.full((repeats, n_learners), np.nan)
    se = np.full((repeats, n_learners), np.nan)

    ckpt = CKPT_DIR / f"{spec.tag}_v2.npz"
    if ckpt.exists():
        old = np.load(ckpt)
        done = min(int(old["done"]), repeats)
        theta[:done] = old["theta"][:done]
        se[:done] = old["se"][:done]
        return theta, se, done

    legacy = CKPT_DIR / f"{spec.tag}.npz"
    if legacy.exists():
        old = np.load(legacy)
        lengths = {len(old[f"{n}|theta"]) for n in LEARNER_NAMES}
        # Ragged lengths mean some learner dropped a repeat; column j would no
        # longer line up with repeat j, so refuse to migrate rather than guess.
        if len(lengths) == 1:
            done = min(lengths.pop(), repeats)
            for j, name in enumerate(LEARNER_NAMES):
                theta[:done, j] = old[f"{name}|theta"][:done]
                se[:done, j] = old[f"{name}|se"][:done]
            print(f"[{spec.tag}] migrated legacy checkpoint ({done} repeats)", flush=True)
            return theta, se, done
        print(f"[{spec.tag}] legacy checkpoint is ragged; starting fresh", flush=True)

    return theta, se, 0


def run_spec_with_checkpoint(data: pd.DataFrame, spec: Spec, repeats: int,
                             folds: int, n_jobs: int) -> dict[str, dict]:
    """All learners for one spec, saving raw draws every CHUNK repeats.

    Restarting the script reloads the saved draws and resumes from where it
    stopped, so a long run survives interruption. Delete the checkpoints folder
    to force a clean rerun.
    """
    keep = [Y, D] + spec.controls + (DISTANCE_INSTRUMENTS if spec.iv else [])
    clean = data[keep].dropna().copy()
    arrays = build_arrays(clean, spec)   # built once, reused by all repeats

    theta, se, done = _load_checkpoint(spec, repeats, len(LEARNER_NAMES))
    if done:
        print(f"[{spec.tag}] resuming from repeat {done}", flush=True)

    ckpt = CKPT_DIR / f"{spec.tag}_v2.npz"
    t0 = time.time()
    cursor = done
    while cursor < repeats:
        stop = min(cursor + CHUNK, repeats)
        # Seeds are spaced far apart so no two repeats share a partition, and
        # each seed is tied to its repeat index -> order-independent results.
        seeds = [RANDOM_STATE + 10_000 * r for r in range(cursor, stop)]
        batch = Parallel(n_jobs=n_jobs, backend="loky", batch_size=1)(
            delayed(_repeat_worker)(arrays, s, folds) for s in seeds)
        for offset, (th, sd) in enumerate(batch):
            theta[cursor + offset] = th
            se[cursor + offset] = sd
        cursor = stop
        np.savez(ckpt, theta=theta, se=se, done=np.array(cursor),
                 names=np.array(LEARNER_NAMES))
        print(f"[{spec.tag}] repeat {cursor}/{repeats}  {time.time() - t0:.0f}s",
              flush=True)

    return {name: aggregate_median(theta[:, j], se[:, j])
            for j, name in enumerate(LEARNER_NAMES)}


def original_estimate(data: pd.DataFrame, spec: Spec) -> tuple[float, float, int]:
    """Benchmark for a spec: our own replication of the corresponding column."""
    if spec.iv:
        result = tsls_conventional(data, Y, D, spec.controls, DISTANCE_INSTRUMENTS)
    else:
        clean = data[[Y, D] + spec.controls].dropna()
        result = ols_conventional(clean[Y].astype(float),
                                  add_constant(clean[[D] + spec.controls].astype(float)))
    return float(result["coef"][D]), float(result["se"][D]), int(result["n"])


# ================================================================= PART 2b ===
# Coefficient-equality test:  H0: theta_ML = theta_Original
#     t = (theta_ML - theta_Orig) / sqrt(SE_ML^2 + SE_Orig^2)
# The pooled denominator credits the benchmark with its own sampling error.
# Cov(theta_ML, theta_Orig) > 0 (same sample) is set to zero, which inflates
# SE_diff and makes the test conservative against finding a difference.
def add_difference_test(results: pd.DataFrame) -> pd.DataFrame:
    is_ml = results["learner"] != "Original"
    results.loc[is_ml, "se_diff"] = np.sqrt(
        results.loc[is_ml, "se"] ** 2 + results.loc[is_ml, "original_se"] ** 2)
    results.loc[is_ml, "t_vs_original"] = (
        results.loc[is_ml, "ml_minus_original"] / results.loc[is_ml, "se_diff"])
    results.loc[is_ml, "p_vs_original"] = (
        results.loc[is_ml, "t_vs_original"].apply(p_from_t))
    results.loc[is_ml, "diff_stars"] = (
        results.loc[is_ml, "p_vs_original"].apply(stars))
    # Kept for reference only: the naive denominator that ignores SE_Orig.
    results.loc[is_ml, "t_vs_original_seml_only"] = (
        results.loc[is_ml, "ml_minus_original"] / results.loc[is_ml, "se"])
    return results


def run_extension(df: pd.DataFrame, repeats: int, folds: int,
                  n_jobs: int) -> pd.DataFrame:
    """PART 2 driver: original benchmark + ten learners for each of four specs."""
    print("\n" + "=" * 78)
    print("PART 2  DML EXTENSION")
    print("=" * 78)
    print(f"parallel workers: {n_jobs if n_jobs > 0 else os.cpu_count()}")
    rows = []
    for spec in ALL_SPECS:
        data = restricted_sample(df) if spec.sample_name == "restricted" else df.copy()
        orig_coef, orig_se, n = original_estimate(data, spec)
        orig_t = orig_coef / orig_se
        orig_p = p_from_t(orig_t)
        rows.append({
            "panel": spec.panel, "column": spec.column, "sample": spec.sample_name,
            "estimator": "Original IV" if spec.iv else "Original OLS",
            "learner": "Original", "coef": orig_coef, "se": orig_se, "t": orig_t,
            "p": orig_p, "stars": stars(orig_p), "original_coef": orig_coef,
            "original_se": orig_se, "ml_minus_original": 0.0, "n": n,
            "valid_repeats": np.nan, "repeat_sd": np.nan,
        })
        print(f"\n{spec.panel}: {spec.column} -- {spec.note}")
        print(f"  original {orig_coef:+.4f} ({orig_se:.4f}), n={n}; "
              f"{folds}-fold x {repeats} repeats, median aggregation", flush=True)
        estimates = run_spec_with_checkpoint(data, spec, repeats, folds, n_jobs)
        for learner_name, est in estimates.items():
            rows.append({
                "panel": spec.panel, "column": spec.column, "sample": spec.sample_name,
                "estimator": "ML-IV" if spec.iv else "ML-OLS", "learner": learner_name,
                "coef": est["ml_coef"], "se": est["ml_se"], "t": est["ml_t"],
                "p": est["ml_p"], "stars": est["stars"],
                "original_coef": orig_coef, "original_se": orig_se,
                # signed gap vs the original coefficient, for the comparison test
                "ml_minus_original": est["ml_coef"] - orig_coef, "n": n,
                "valid_repeats": est["valid_repeats"], "repeat_sd": est["repeat_sd"],
            })

    results = add_difference_test(pd.DataFrame(rows))

    results.to_csv(OUT_DIR / "all_results.csv", index=False)
    results[results["panel"] == "OLS vs ML-OLS"].to_csv(OUT_DIR / "ols_results.csv", index=False)
    results[results["panel"] == "IV vs ML-IV"].to_csv(OUT_DIR / "iv_results.csv", index=False)
    return results


# =============================================================================
# PART 3.  Output: LaTeX tables and coefficient plots
# =============================================================================

def format_coef_se(row: pd.Series) -> str:
    return f"{row['coef']:.3f}{row['stars']} ({row['se']:.3f})"


def export_latex_panel(results: pd.DataFrame, panel: str, filename: str,
                       caption: str) -> None:
    """One LaTeX table per panel: rows = Original + 10 learners, cols = specs."""
    panel_df = results[results["panel"] == panel].copy()
    columns = list(panel_df["column"].drop_duplicates())
    learners = ["Original"] + LEARNER_NAMES
    rows = []
    for learner in learners:
        row = {"Estimator": learner}
        for col in columns:
            match = panel_df[(panel_df["learner"] == learner) & (panel_df["column"] == col)]
            row[col] = format_coef_se(match.iloc[0]) if not match.empty else ""
        rows.append(row)
    latex = pd.DataFrame(rows).to_latex(
        index=False, escape=False, caption=caption,
        label=f"tab:{filename.replace('.tex', '')}",
        column_format="l" + "c" * len(columns))
    notes = (
        "\\\\[-0.5em]\n"
        "\\multicolumn{" + str(len(columns) + 1) + "}{p{0.95\\linewidth}}{\\footnotesize "
        "Notes: Entries are coefficients on log slave exports per unit land area. "
        "Original rows reproduce Nunn (2008) with the conventional standard errors "
        "printed in the paper. ML rows use 5-fold cross-fitting repeated 500 times, "
        "with one common fold partition shared by all nuisance functions within a "
        "repeat. Coefficients are medians across repeats; standard errors use the "
        "median aggregation of Chernozhukov et al. (2018), $\\widehat{se} = "
        "\\sqrt{\\mathrm{med}_r[se_r^2 + (\\hat\\theta_r - \\tilde\\theta)^2]}$, "
        "combining within-split uncertainty and split variation. "
        "* p<0.10, ** p<0.05, *** p<0.01.}\\\\\n")
    (OUT_DIR / filename).write_text(latex.replace("\\end{tabular}", notes + "\\end{tabular}"))


# ------------------------------------------- standalone difference-test table --
def export_difference_table(results: pd.DataFrame, panel: str, filename: str,
                            caption: str) -> None:
    """Coefficient-equality test, on its own -- no ML coefficients in this table.

    Rows are the ten learners; every spec column gets a difference cell
    (with the pooled SE in brackets) and its t-statistic.
    """
    panel_df = results[results["panel"] == panel].copy()
    columns = list(panel_df["column"].drop_duplicates())
    ncols = 1 + 2 * len(columns)
    label = filename.replace(".tex", "")

    lines = [
        "\\begin{table}[htbp]",
        "\\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{tab:{label}}}",
        "\\begin{tabular}{l" + "cc" * len(columns) + "}",
        "\\toprule",
        " & " + " & ".join(f"\\multicolumn{{2}}{{c}}{{{c}}}" for c in columns) + " \\\\",
        " ".join(f"\\cmidrule(lr){{{2 + 2 * i}-{3 + 2 * i}}}"
                 for i in range(len(columns))),
        "Learner & " + " & ".join(
            ["$\\hat\\theta_{ML}-\\hat\\theta_{Orig}$ & $t$"] * len(columns)) + " \\\\",
        "\\midrule",
    ]

    # Benchmark row: the value every difference in the table is measured against.
    bench = []
    for col in columns:
        row = panel_df[(panel_df["column"] == col)
                       & (panel_df["learner"] == "Original")].iloc[0]
        bench.append(f"\\multicolumn{{2}}{{c}}{{${row['coef']:.3f}$ ({row['se']:.3f})}}")
    lines.append("\\textit{Benchmark} $\\hat\\theta_{Orig}$ (SE) & "
                 + " & ".join(bench) + " \\\\")
    lines.append("\\midrule")

    for learner in LEARNER_NAMES:
        cells = []
        for col in columns:
            match = panel_df[(panel_df["learner"] == learner)
                             & (panel_df["column"] == col)]
            if match.empty:
                cells += ["", ""]
                continue
            r = match.iloc[0]
            # isinstance guard: an empty star string round-trips through CSV as NaN
            star = r["diff_stars"] if isinstance(r["diff_stars"], str) else ""
            sup = f"^{{{star}}}" if star else ""
            cells.append(f"${r['ml_minus_original']:+.3f}{sup}$ "
                         f"[{r['se_diff']:.3f}]")
            cells.append(f"{r['t_vs_original']:.2f}")
        lines.append(f"{learner} & " + " & ".join(cells) + " \\\\")

    lines.append("\\bottomrule")
    lines.append(
        "\\multicolumn{" + str(ncols) + "}{p{0.95\\linewidth}}{\\footnotesize "
        "Notes: This table reports ONLY the test of $H_0:\\theta_{ML}="
        "\\theta_{Orig}$; the underlying coefficients are in the companion table. "
        "Each cell gives $\\hat\\theta_{ML}-\\hat\\theta_{Orig}$ with the pooled "
        "standard error $\\sqrt{SE_{ML}^2+SE_{Orig}^2}$ in brackets, and the "
        "adjacent column the implied $t=(\\hat\\theta_{ML}-\\hat\\theta_{Orig})/"
        "\\sqrt{SE_{ML}^2+SE_{Orig}^2}$; $p$-values are two-sided normal. "
        "$\\hat\\theta_{Orig}$ is our replication of the corresponding published "
        "column, with the conventional (homoskedastic) standard error used in the "
        "paper; $SE_{ML}$ is the median-aggregated DML standard error over 500 "
        "repeats. Both estimators are computed on the same countries, so "
        "$\\mathrm{Cov}(\\hat\\theta_{ML},\\hat\\theta_{Orig})>0$; setting it to "
        "zero inflates the denominator and makes the test conservative against "
        "rejecting equality. A non-significant entry should therefore be read as "
        "no detectable divergence from the published estimate under this "
        "convention. * p<0.10, ** p<0.05, *** p<0.01.}\\\\")
    lines += ["\\end{tabular}", "\\end{table}", ""]

    (OUT_DIR / filename).write_text("\n".join(lines))


def plot_panel(results: pd.DataFrame, panel: str, filename: str) -> None:
    """Coefficient plot: black cross = original, colored dots = learners, 95% CIs."""
    panel_df = results[results["panel"] == panel].copy()
    columns = list(panel_df["column"].drop_duplicates())
    learners = LEARNER_NAMES
    palette = {
        "Original": "#111111", "Linear": "#d62728", "Ridge": "#2ca02c",
        "LASSO": "#f1c40f", "Elastic Net": "#17becf", "Polynomial Ridge": "#9467bd",
        "SVR": "#8c564b", "Kernel Ridge": "#e377c2", "Random Forest": "#1f77b4",
        "Extra Trees": "#7f7f7f", "Gradient Boosting": "#ff7f0e",
    }
    fig, ax = plt.subplots(figsize=(11, 6.2))
    # +-0.22 keeps a visible gap between the two column groups (spacing is 1.0),
    # so dots from different specs are not mistaken for one cluster.
    offsets = np.linspace(-0.22, 0.22, len(learners))
    for y, col in enumerate(columns):
        orig = panel_df[(panel_df["column"] == col) & (panel_df["learner"] == "Original")].iloc[0]
        ax.errorbar(orig["coef"], y, xerr=1.96 * orig["se"], fmt="x",
                    color=palette["Original"], elinewidth=2.2, capsize=3, markersize=8,
                    label="Original" if y == 0 else None, zorder=5)
        for offset, learner in zip(offsets, learners):
            row = panel_df[(panel_df["column"] == col) & (panel_df["learner"] == learner)].iloc[0]
            ax.errorbar(row["coef"], y + offset, xerr=1.96 * row["se"], fmt="o",
                        color=palette[learner], elinewidth=1.4, capsize=2,
                        markersize=4.3, alpha=0.9, label=learner if y == 0 else None)
        if y < len(columns) - 1:  # separator between column groups
            ax.axhline(y + 0.5, color="#cccccc", linewidth=0.8, zorder=0)
    ax.axvline(0, color="#666666", linestyle=":", linewidth=1)
    ax.set_yticks(range(len(columns)))
    ax.set_yticklabels(columns)
    ax.invert_yaxis()
    ax.set_xlabel("Coefficient on log slave exports (95% CI)")
    ax.set_title(f"Nunn (2008): {panel} - 5-fold x 500-repeat median DML", weight="bold")
    ax.grid(axis="x", color="#dddddd", linewidth=0.8)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=True)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=220, bbox_inches="tight")
    plt.close(fig)


def print_difference_summary(results: pd.DataFrame) -> None:
    """Console version of the two difference tables."""
    print("\n" + "=" * 78)
    print("PART 2b  H0: theta_ML = theta_Original   "
          "t = diff / sqrt(SE_ML^2 + SE_Orig^2)")
    print("=" * 78)
    ml = results[results["learner"] != "Original"]
    for panel in ["OLS vs ML-OLS", "IV vs ML-IV"]:
        print(f"\n{panel}")
        sub = ml[ml["panel"] == panel]
        for col in sub["column"].drop_duplicates():
            block = sub[sub["column"] == col]
            orig = block.iloc[0]["original_coef"]
            print(f"  {col}  (benchmark {orig:+.4f})")
            print(f"    {'learner':<19}{'diff':>9}{'se_diff':>10}{'t':>8}{'p':>9}")
            for _, r in block.iterrows():
                print(f"    {r['learner']:<19}{r['ml_minus_original']:>+9.4f}"
                      f"{r['se_diff']:>10.4f}{r['t_vs_original']:>8.2f}"
                      f"{r['p_vs_original']:>9.3f} {r['diff_stars']}")


# =============================================================================
# main
# =============================================================================

def main() -> None:
    df = load_data()

    # PART 1: replicate first, always. It validates the data and prints a
    # PASS/DIFF flag for every published cell.
    replicate_tables(df)

    # PART 2: extend the four selected columns.
    results = run_extension(df, repeats=REPEATS, folds=FOLDS, n_jobs=N_JOBS)

    # PART 2b: console summary of the equality test.
    print_difference_summary(results)

    # PART 3: tables and figures.
    export_latex_panel(results, "OLS vs ML-OLS", "ols_vs_mlols_table.tex",
                       "Original OLS and ML-OLS (DML-PLR) Estimates")
    export_latex_panel(results, "IV vs ML-IV", "iv_vs_mliv_table.tex",
                       "Original IV and ML-IV (DML-PLIV) Estimates")
    # Two standalone equality-test tables, kept separate from the coefficients.
    export_difference_table(
        results, "OLS vs ML-OLS", "ols_difference_test.tex",
        "Test of $H_0:\\theta_{ML}=\\theta_{Orig}$, OLS Specifications")
    export_difference_table(
        results, "IV vs ML-IV", "iv_difference_test.tex",
        "Test of $H_0:\\theta_{ML}=\\theta_{Orig}$, IV Specifications")
    plot_panel(results, "OLS vs ML-OLS", "ols_vs_mlols_coefficient_plot.png")
    plot_panel(results, "IV vs ML-IV", "iv_vs_mliv_coefficient_plot.png")
    print(f"\nSaved outputs in {OUT_DIR}")


if __name__ == "__main__":
    main()

Data loaded from GitHub: 52 rows, 39 columns

PART 1  REPLICATION OF NUNN (2008)

Table III - OLS, dependent variable ln real per capita GDP 2000
 col      coef       se    n            published   check
   1   -0.1119   0.0238   52      -0.112 (0.024)   PASS
   2   -0.0762   0.0291   52      -0.076 (0.029)   PASS
   3   -0.1080   0.0377   42      -0.108 (0.037)   PASS
   4   -0.0854   0.0349   52      -0.085 (0.035)   PASS
   5   -0.1032   0.0339   52      -0.103 (0.034)   PASS
   6   -0.1277   0.0349   42      -0.128 (0.034)   PASS

Table IV - 2SLS, four distance instruments
 col      coef       se    n   1stF            published   check
   1   -0.2079   0.0530   52   4.55      -0.208 (0.053)   PASS
       first stage -> atlantic: -1.31 (0.357), indian: -1.10 (0.380), saharan: -2.43 (0.823), red: -0.00 (0.710)
   2   -0.2014   0.0472   52   5.10      -0.201 (0.047)   PASS
       first stage -> atlantic: -1.74 (0.425), indian: -1.43 (0.531), saharan: -3.00 (1.049), red: -0.15 (0.813)